## ENEM Caxambu 2021–2024 — Consolidação longitudinal e agregações

Objetivo: consolidar os recortes de Minas Gerais entre 2021 e 2024, harmonizar variáveis de participantes e resultados, gerar tabelas agregadas para análise exploratória e dashboard, e construir bases integradas em nível agregado para comparações entre perfil socioeconômico e desempenho.

Entradas:

* DADOS_TRAT_CAX_2021
* DADOS_TRAT_CAX_2022
* DADOS_TRAT_CAX_2023
* DADOS_TRAT_CAX_2024
* RESULTADOS_TRAT_CAX_2024

Saídas esperadas:

* base consolidada de Caxambu 2021–2024 em nível candidato;
* base consolidada de resultados;
* base agregada socioeconômica;
* base agregada da base de resultados;
* base merged agregada;
* amostra estratificada de resultados;
* base merged numérica.
  
Observação metodológica: nos anos de 2021 a 2023, as informações socioeconômicas e de desempenho estão presentes na mesma base tratada. Em 2024, participantes e resultados foram divulgados separadamente, o que impede associação individual. Assim, a integração entre perfil socioeconômico e desempenho em 2024 é feita apenas em nível agregado.

Observação (GitHub): esta etapa trabalha apenas sobre arquivos Parquet previamente tratados e harmonizados nas etapas anteriores.

### 1) Ambiente e imports

In [1]:
import sys
from pathlib import Path

# Permite importar o pacote `src/` a partir do diretório do projeto.
ROOT_PATH = Path().resolve().parents[1]  # notebooks/00_preprocessamento -> projeto
if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

import re
import numpy as np
import pandas as pd


from src.config import (
    DADOS_TRAT_CAX_2021,
    DADOS_TRAT_CAX_2022,
    DADOS_TRAT_CAX_2023,
    DADOS_TRAT_CAX_2024,
    RESULTADOS_TRAT_CAX_2024,
    DADOS_CAX_21_23,
    DADOS_AGG_CAX_21_23, 
    MERGED_CAX,
    MERGED_CAX_NUM,
    DADOS_AGG_CAX,
    RESULTADOS_AGG_CAX,
    DADOS_UNI_CAX,
    RESULTADOS_UNI_CAX,
)


from src.preprocessamento.agregacoes import  (
    agregar_perfil_socioeconomico,
    agrupar_notas, 
    amostrar_por_percentil_original, 
    categoria_ordenada_para_numero,
)

from src.preprocessamento.categorias import ORDEM_OCUPACAO, ORDEM_PAIS_ESCOLARIDADE


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


### 2) Leitura e consolidação anual

Nesta etapa são carregados os recortes anuais de Caxambu já tratados. Em seguida, é criada a variável ano e os arquivos são concatenados para formar uma base longitudinal única, utilizada nas análises exploratórias, nas agregações.

In [2]:
df_cx_21  = pd.read_parquet(DADOS_TRAT_CAX_2021)
df_cx_22  = pd.read_parquet(DADOS_TRAT_CAX_2022)
df_cx_23  = pd.read_parquet(DADOS_TRAT_CAX_2023)
df_cx_24  = pd.read_parquet(DADOS_TRAT_CAX_2024)
df_cx_r_24  = pd.read_parquet(RESULTADOS_TRAT_CAX_2024)


In [3]:
df_cx_21['ano'] = '2021'
df_cx_22['ano'] = '2022'
df_cx_23['ano'] = '2023'
df_cx_24['ano'] = '2024'
df_cx_r_24['ano'] = '2024'

df_cx_21_23 = pd.concat([df_cx_21, df_cx_22, df_cx_23], ignore_index=True)

for base in [df_cx_21_23, df_cx_24, df_cx_r_24]:
    base["ano"] = base["ano"].astype(str)
    base=base.drop(columns=["municipio"], errors="ignore")

In [4]:
df_cx_21_23.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,escola,municipio,uf,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo,renda_media,nota_media,ano
0,até 19,feminino,solteiro,branca,pública,Caxambu,MG,437.60,517.20,541.90,512.10,espanhol,720.00,superior,médio,médio/tec,médio/tec,3,1 a 3,0,1,1,2,3,2,0.25,2.00,545.76,2021
1,até 19,masculino,solteiro,branca,pública,Caxambu,MG,620.30,602.20,573.20,623.30,inglês,720.00,até fund,até fund,manual/tec,médio/tec,3,1 a 3,0,1,1,2,3,2,0.35,2.00,627.80,2021
2,até 19,feminino,solteiro,branca,não informada,Caxambu,MG,509.40,564.70,528.20,473.10,inglês,840.00,médio,médio,manual/tec,básico,4,1 a 3,0,1,1,1,3,1,0.25,2.00,583.08,2021


### 3) Harmonização entre base de dados e base de resultados

Como os anos de 2021 a 2023 contêm informações socioeconômicas e de desempenho na mesma base, enquanto 2024 traz participantes e resultados em arquivos separados, foi necessário harmonizar a estrutura dos dados em dois blocos:

* base de resultados, contendo notas e métricas de presença;
* base de dados, contendo características socioeconômicas e demográficas.

Essa separação permite gerar agregações compatíveis entre os anos e construir posteriormente bases integradas em nível agregado.

In [5]:
colunas_notas = [col for col in df_cx_21_23.columns if col.startswith("nota")]
colunas_notas.append("lingua")

colunas_resultados = [
    "ano", "escola", "uf", "municipio",
    "nota_cn", "nota_ch", "nota_lc", "nota_mt",
    "lingua", "nota_redacao", "nota_media"
]

df_result_21_23 = df_cx_21_23[colunas_resultados].copy()

df_resultados = pd.concat([df_result_21_23, df_cx_r_24], ignore_index=True)


df_dados_21_23 = df_cx_21_23.drop(columns=colunas_notas, errors="ignore").copy()
df_dados = pd.concat([df_dados_21_23, df_cx_24], ignore_index=True)

In [6]:
df_resultados.head(3)

,ano,escola,uf,municipio,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,nota_media
0,2021,pública,MG,Caxambu,437.60,517.20,541.90,512.10,espanhol,720.00,545.76
1,2021,pública,MG,Caxambu,620.30,602.20,573.20,623.30,inglês,720.00,627.80
2,2021,não informada,MG,Caxambu,509.40,564.70,528.20,473.10,inglês,840.00,583.08


### 4) Agregação da base de resultados


A base de resultados é agregada para sintetizar o desempenho médio dos grupos e os indicadores de comparecimento às provas. Essa agregação permite comparar regiões e tipos de escola sem depender de vínculo individual entre participantes e resultados.

São calculados:

* médias de notas por área e da nota_media;
* maiores notas observadas por área;
* desvio-padrão da nota_media;
* quantidade de inscritos;
* faltosos e taxas de presença por dia de prova.

In [7]:
df_cx_r_24_agg = agrupar_notas(df_cx_r_24, incluir_municipio=True)
df_resultados_agg = agrupar_notas(df_resultados, incluir_municipio=True)

df_resultados_agg.head(3)


,ano,uf,escola,municipio,nota_media,nota_cn,nota_cn_max,nota_ch,nota_ch_max,nota_lc,nota_lc_max,nota_mt,nota_mt_max,nota_redacao,nota_redacao_max,desvio_padrao,participantes,inscritos,faltosos_dia1,faltosos_dia2,presentes_dia1,presentes_dia2,taxa_presenca_dia1,taxa_presenca_dia2
0,2021,MG,pública,Caxambu,526.67,479.99,705.10,507.93,757.50,487.67,659.30,533.56,888.10,628.41,960.00,85.82,221,221,26,31,195,190,0.88,0.86
1,2021,MG,não informada,Caxambu,544.66,502.18,712.00,528.71,758.40,509.90,700.20,557.51,845.60,630.95,980.00,91.43,567,567,147,164,420,403,0.74,0.71
2,2021,MG,privada,Caxambu,631.32,559.75,773.40,583.78,750.40,560.68,691.20,660.21,870.30,801.46,980.00,81.49,42,42,1,2,41,40,0.98,0.95


### 5) Preparação da versão numérica para modelagem

Para a etapa de modelagem, também foi criada uma versão numérica da base consolidada. Nessa versão, variáveis ordinais relacionadas ao capital familiar — como escolaridade e ocupação dos pais — são convertidas em códigos numéricos, preservando a ordem substantiva definida no pré-processamento.

Para o gráfico de correlação, na etapa EDA e Dashboard a Variáve

Essa conversão é útil para:

* correlação;
* análises de sensibilidade;
*construção de bases merged para modelagem;
* consolidação final do dataset voltado a ML.

### 5.1) Confirmação da ordenação categórica

In [8]:
print(df_dados['ocup_pai'].unique())
print(df_dados['escolaridade_pai'].unique())

['médio/tec', 'manual/tec', 'superior', 'desconhecida', 'básico', 'rural']
Categories (6, object): ['desconhecida' < 'rural' < 'básico' < 'manual/tec' < 'médio/tec' < 'superior']
['superior', 'até fund', 'médio', 'pós-grad', 'desconhecida', 'não estudou']
Categories (6, object): ['desconhecida' < 'não estudou' < 'até fund' < 'médio' < 'superior' < 'pós-grad']


### 5.2) Conversão para códigos numéricos

Transformação da variável escola para análise de correlação

A variável escola é originalmente categórica nominal, representando o tipo de instituição de ensino médio frequentada pelos participantes:

* pública
* privada
* não informada

Para viabilizar sua inclusão em análises de correlação, foi adotada uma codificação ordinal controlada, baseada na relação empírica observada entre tipo de escola e desempenho médio.

Observou-se que:

* candidatos de escolas públicas apresentam, em média, menores notas;
* candidatos que não informaram o tipo de escola apresentam desempenho intermediário;
* candidatos de escolas privadas apresentam, em média, maiores notas.

Com base nesse padrão, definiu-se a seguinte ordenação:

* pública → 0
* não informada → 1
* privada → 2

Essa transformação não implica que a variável seja quantitativa, mas permite sua utilização como uma aproximação ordinal em análises exploratórias, especialmente em matrizes de correlação.

Interpretação

A codificação deve ser interpretada como um gradiente aproximado de condições educacionais associadas ao tipo de escola, e não como uma medida contínua.

Limitação metodológica

Essa abordagem pressupõe uma relação monotônica entre as categorias, o que pode não capturar completamente a heterogeneidade interna de cada grupo.

Por esse motivo, na etapa de modelagem, a variável escola é tratada como categórica, sendo transformada por meio de one-hot encoding, evitando a imposição de uma estrutura ordinal artificial no modelo.

In [9]:
df_dados_num = df_dados.copy()
df_21_23_num = df_dados_21_23.copy()

for col in ["ocup_pai", "ocup_mae", "escolaridade_pai", "escolaridade_mae"]:
    df_dados_num[col] = categoria_ordenada_para_numero(df_dados_num[col])
    df_21_23_num[col] = categoria_ordenada_para_numero(df_21_23_num[col])

df_dados_num['escola_num']=categoria_ordenada_para_numero(df_dados_num['escola'])
df_21_23_num['escola_num']=categoria_ordenada_para_numero(df_21_23_num['escola'])

df_dados_num['escolaridade_pais_media'] = (df_dados_num['escolaridade_pai'] + df_dados_num['escolaridade_mae']) / 2
df_dados_num['ocup_pais_media'] = (df_dados_num['ocup_mae'] + df_dados_num['ocup_pai']) / 2
df_dados_num.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,escola,municipio,uf,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo,renda_media,ano,escola_num,escolaridade_pais_media,ocup_pais_media
0,até 19,feminino,solteiro,branca,pública,Caxambu,MG,4,3,4,4,3,1 a 3,0,1,1,2,3,2,0.25,2.00,2021,0,3.50,4.00
1,até 19,masculino,solteiro,branca,pública,Caxambu,MG,2,2,3,4,3,1 a 3,0,1,1,2,3,2,0.35,2.00,2021,0,2.00,3.50
2,até 19,feminino,solteiro,branca,não informada,Caxambu,MG,3,3,3,2,4,1 a 3,0,1,1,1,3,1,0.25,2.00,2021,1,3.00,2.50


### 6) Agregação da base socioeconômica

A base socioeconômica é agregada por ano, região e características demográficas e familiares. O objetivo é construir perfis médios de grupos populacionais comparáveis ao longo do tempo, preservando tanto o tamanho dos grupos quanto indicadores de infraestrutura, renda e composição domiciliar.

In [10]:
cols_agg = [
    "ano",
    "uf",
    "municipio",
    "escola",
    "sexo",
    "cor_raca",
    "faixa_etaria",
    "sal_min",
    "escolaridade_pai",
    "escolaridade_mae",
    "ocup_pai",
    "ocup_mae",
]

df_dados_agg = agregar_perfil_socioeconomico(df_dados, cols_agg)
df_dados_agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2688 entries, 0 to 2687
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   ano               2688 non-null   object  
 1   uf                2688 non-null   category
 2   municipio         2688 non-null   category
 3   escola            2688 non-null   category
 4   sexo              2688 non-null   category
 5   cor_raca          2688 non-null   category
 6   faixa_etaria      2688 non-null   category
 7   sal_min           2688 non-null   category
 8   escolaridade_pai  2688 non-null   category
 9   escolaridade_mae  2688 non-null   category
 10  ocup_pai          2688 non-null   category
 11  ocup_mae          2688 non-null   category
 12  participantes     2688 non-null   int64   
 13  cel               2688 non-null   Float64 
 14  comptdr           2688 non-null   Float64 
 15  n_pessoas_resd    2688 non-null   Float64 
 16  renda_media       2688 n

### 7) Base merged agregada
Para analisar conjuntamente indicadores socioeconômicos e desempenho educacional, foi construída uma base integrada em nível agregado. Como a associação individual não é possível em 2024, a integração é realizada por chaves comuns (ano, uf, regiao, escola), unindo a agregação da base de dados à agregação da base de resultados.

In [11]:
cols_merged = ["ano", "uf", "escola"]

df_dados_merged = agregar_perfil_socioeconomico(df_dados_num, cols_merged, incluir_escola_num_pais_media=True, )

df_merged = pd.merge(
    df_resultados_agg,
    df_dados_merged,
    on=["ano", "uf", "escola"],
    how="outer",
    suffixes=("_notas", "_renda"),
)


df_merged = df_merged.drop(columns=["participantes_notas"], errors="ignore")
df_merged= df_merged.rename(columns={"participantes_renda": "participantes"})
df_merged.head(3)

,ano,uf,escola,municipio,nota_media,nota_cn,nota_cn_max,nota_ch,nota_ch_max,nota_lc,nota_lc_max,nota_mt,nota_mt_max,nota_redacao,nota_redacao_max,desvio_padrao,inscritos,faltosos_dia1,faltosos_dia2,presentes_dia1,presentes_dia2,taxa_presenca_dia1,taxa_presenca_dia2,participantes,cel,comptdr,n_pessoas_resd,renda_media,indice_consumo,escola_num,escolaridade_pais_media,ocup_pais_media
0,2021,MG,pública,Caxambu,526.67,479.99,705.10,507.93,757.50,487.67,659.30,533.56,888.10,628.41,960.00,85.82,221,26,31,195,190,0.88,0.86,221,2.57,0.78,3.79,2.45,0.26,0.00,2.69,2.22
1,2021,MG,não informada,Caxambu,544.66,502.18,712.00,528.71,758.40,509.90,700.20,557.51,845.60,630.95,980.00,91.43,567,147,164,420,403,0.74,0.71,567,2.68,0.91,3.57,3.14,0.28,1.00,2.87,2.52
2,2021,MG,privada,Caxambu,631.32,559.75,773.40,583.78,750.40,560.68,691.20,660.21,870.30,801.46,980.00,81.49,42,1,2,41,40,0.98,0.95,42,3.12,1.81,3.43,5.05,0.40,2.00,3.67,3.48


### 8) Base histórica agregada 2021–2023

Além da consolidação longitudinal completa, foi gerada uma agregação específica para o período 2021–2023, em que informações socioeconômicas e notas estavam presentes no mesmo arquivo-base. Isso permite comparações históricas diretas sem necessidade de integração entre arquivos distintos.

In [12]:
cols_agg=['ano','uf', 'municipio', 'escola', 'sal_min','cor_raca', 'faixa_etaria','escolaridade_pai', 'escolaridade_mae', 'ocup_pai',
       'ocup_mae']

for col in cols_agg:
    if df_cx_21_23[col].dtype.name == 'category':
        df_cx_21_23[col] = df_cx_21_23[col].cat.remove_unused_categories()

df_agg_21_23 = df_cx_21_23.groupby(cols_agg, observed=True, dropna=False).agg(
    participantes=("municipio", "count"),
    cel=("cel", lambda x: round(x.mean())),
    comptdr=("comptdr", lambda x: round(x.mean())),
    n_pessoas_resd=("n_pessoas_resd", lambda x: round(x.mean())),
    renda_media=("renda_media","mean"),
    indice_consumo=("indice_consumo", "mean"),
    nota_cn=("nota_cn", "mean"),
    nota_ch=("nota_ch", "mean"),
    nota_lc=("nota_lc", "mean"),
    nota_mt=("nota_mt", "mean"),
    nota_redacao=("nota_redacao", "mean"),
    nota_media=("nota_media", "mean"),
).round(2)

df_agg_21_23 = df_agg_21_23.reset_index()

# Filtrar apenas grupos com participantes
df_agg_21_23 = df_agg_21_23[df_agg_21_23['participantes'] > 0]

df_agg_21_23.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1738 entries, 0 to 1737
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   ano               1738 non-null   object  
 1   uf                1738 non-null   category
 2   municipio         1738 non-null   category
 3   escola            1738 non-null   category
 4   sal_min           1738 non-null   category
 5   cor_raca          1738 non-null   category
 6   faixa_etaria      1738 non-null   category
 7   escolaridade_pai  1738 non-null   category
 8   escolaridade_mae  1738 non-null   category
 9   ocup_pai          1738 non-null   category
 10  ocup_mae          1738 non-null   category
 11  participantes     1738 non-null   int64   
 12  cel               1738 non-null   Int8    
 13  comptdr           1738 non-null   Int8    
 14  n_pessoas_resd    1738 non-null   Int8    
 15  renda_media       1738 non-null   float32 
 16  indice_consumo    1738 n

### 9) Exportação das bases consolidadas — Caxambu

Ao final desta etapa, são exportados os arquivos derivados da consolidação longitudinal dos microdados do ENEM para o município de Caxambu, abrangendo o período 2021–2024.

In [13]:
df_cx_21_23.to_parquet(DADOS_CAX_21_23, index=False)
df_dados.to_parquet(DADOS_UNI_CAX, index=False)
df_resultados.to_parquet(RESULTADOS_UNI_CAX, index=False)
df_dados_agg.to_parquet(DADOS_AGG_CAX, index=False)
df_resultados_agg.to_parquet(RESULTADOS_AGG_CAX, index=False)
df_merged.to_parquet(MERGED_CAX, index=False)
df_agg_21_23.to_parquet(DADOS_AGG_CAX_21_23, index=False)


print("✅ Salvo base Caxambu 2021–2023:", DADOS_CAX_21_23, "| shape:", df_cx_21_23.shape)
print("✅ Salvo base unificada de dados Caxambu 2021–2024:", DADOS_UNI_CAX, "| shape:", df_dados.shape)
print("✅ Salvo base unificada de resultados Caxambu 2021–2024:", RESULTADOS_UNI_CAX, "| shape:", df_resultados.shape)
print("✅ Salvo base socioeconômica agregada Caxambu 2021–2024:", DADOS_AGG_CAX, "| shape:", df_dados_agg.shape)
print("✅ Salvo base de resultados agregada Caxambu 2021–2024:", RESULTADOS_AGG_CAX, "| shape:", df_resultados_agg.shape)
print("✅ Salvo dataframe merged Caxambu 2021–2024:", MERGED_CAX, "| shape:", df_merged.shape)
print("✅ Salvo base agregada histórica Caxambu 2021–2023:", DADOS_AGG_CAX_21_23, "| shape:", df_agg_21_23.shape)

✅ Salvo base Caxambu 2021–2023: E:\ciencias_dados\projetos\projeto_enem_ml\dados\analiticos\dados_21_23_cax.parquet | shape: (2562, 29)
✅ Salvo base unificada de dados Caxambu 2021–2024: E:\ciencias_dados\projetos\projeto_enem_ml\dados\analiticos\dados_uni_cax.parquet | shape: (3644, 22)
✅ Salvo base unificada de resultados Caxambu 2021–2024: E:\ciencias_dados\projetos\projeto_enem_ml\dados\analiticos\resultados_uni_cax.parquet | shape: (3644, 11)
✅ Salvo base socioeconômica agregada Caxambu 2021–2024: E:\ciencias_dados\projetos\projeto_enem_ml\dados\analiticos\dados_agg_cax.parquet | shape: (2688, 18)
✅ Salvo base de resultados agregada Caxambu 2021–2024: E:\ciencias_dados\projetos\projeto_enem_ml\dados\analiticos\resultados_agg_cax.parquet | shape: (12, 24)
✅ Salvo dataframe merged Caxambu 2021–2024: E:\ciencias_dados\projetos\projeto_enem_ml\dados\analiticos\merged_cax.parquet | shape: (12, 32)
✅ Salvo base agregada histórica Caxambu 2021–2023: E:\ciencias_dados\projetos\projeto_ene